# Phase 2-plus — Step 1: LLM Productivity vs. Context Restoration

## Objective
Prove empirically that the Parent-Child Reconstruction (Phase 2, Step 3) has a **direct, measurable impact on LLM answer quality**.

## Method
For each test case, we feed Llama 3.2 (via Ollama, running locally) the **same user question** with two different contexts:
- **Scenario A — Broken Context:** Only the isolated PII chunk (e.g., 48-char password fragment).
- **Scenario B — Restored Context:** The full parent paragraph after reconstruction (e.g., 286-char log entry).

The LLM is asked to answer the user's question based exclusively on the provided context. We compare both answers to demonstrate the productivity difference.

In [1]:
# Cell 1 — Imports, Ollama client, configuration
import json, time, os, textwrap, requests
from dataclasses import dataclass, field

BASE = os.path.dirname(os.path.abspath("__file__"))
RESULTS_DIR = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph2")
CHUNK_FILE = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph1", "chunk_results.json")

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.2"

# Verify Ollama is running
resp = requests.get("http://localhost:11434/api/tags")
models = [m["name"] for m in resp.json()["models"]]
print(f"Ollama connected. Available models: {models}")
assert any(MODEL in m for m in models), f"{MODEL} not found!"

def llm_generate(prompt: str, temperature: float = 0.1, max_tokens: int = 300) -> tuple[str, float]:
    """Call Ollama Llama 3.2 and return (response_text, latency_seconds)."""
    t0 = time.time()
    resp = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens,
        }
    })
    latency = time.time() - t0
    return resp.json()["response"].strip(), latency

# Quick test
test_resp, test_lat = llm_generate("Say 'hello' and nothing else.")
print(f"LLM test: \"{test_resp}\" ({test_lat:.1f}s)")
print("Ready.")

Ollama connected. Available models: ['llama3.2:latest']


LLM test: "Hello." (2.6s)
Ready.


In [2]:
# Cell 2 — Load chunk data and build parent-child pairs

with open(CHUNK_FILE, "r", encoding="utf-8") as f:
    all_strategies = json.load(f)

custom = all_strategies["custom_rbac"]
fixed = all_strategies["fixed"]

def get_chunk(strategy_data: dict, source_file: str, chunk_id: str) -> dict:
    for ch in strategy_data[source_file]:
        if ch["metadata"]["chunk_id"] == chunk_id:
            return ch
    raise ValueError(f"Chunk {chunk_id} not found in {source_file}")

# Define the 5 test cases with their parent-child pairs
@dataclass
class TestCase:
    tid: str
    query: str
    child_source: str
    child_id: str
    parent_source: str
    parent_id: str
    pii_type: str
    description: str

TESTS = [
    TestCase("T1",
             "What is the admin override password for the backend server, and in what context was it used?",
             "server_logs_witty_backend.txt", "custom_005",
             "server_logs_witty_backend.txt", "fixed_001",
             "password",
             "Password — isolated 48ch vs full 286ch log entry"),
    TestCase("T2",
             "What is the IBAN for payments to the manufacturer, and what are the payment conditions?",
             "distribution-contract-2026.docx", "custom_004",
             "distribution-contract-2026.docx", "fixed_002",
             "iban",
             "IBAN — isolated 55ch vs full 310ch contract clause"),
    TestCase("T3",
             "What IP address was involved in the security incident, and what happened?",
             "server_logs_witty_backend.txt", "custom_002",
             "server_logs_witty_backend.txt", "fixed_000",
             "ip_address",
             "IP address — isolated 21ch vs full 462ch log block"),
    TestCase("T4",
             "What is the email and billing information for client CLI-001?",
             "clients-and-billings.xlsx", "custom_001",
             "clients-and-billings.xlsx", "fixed_000",
             "email",
             "Client email — isolated 104ch row vs full 433ch spreadsheet"),
    TestCase("T5",
             "Who is the distributor representative named in the contract, and what company do they represent?",
             "distribution-contract-2026.docx", "custom_000",
             "distribution-contract-2026.docx", "fixed_000",
             "person_name",
             "Person name — isolated 250ch vs full 346ch contract header"),
]

# Load actual text for each pair
for tc in TESTS:
    child = get_chunk(custom, tc.child_source, tc.child_id)
    parent = get_chunk(fixed, tc.parent_source, tc.parent_id)
    tc.child_text = child["page_content"]
    tc.parent_text = parent["page_content"]

print(f"Loaded {len(TESTS)} test cases:\n")
for tc in TESTS:
    print(f"  {tc.tid}: {tc.description}")
    print(f"       Child:  {len(tc.child_text)} chars | Parent: {len(tc.parent_text)} chars")
    print(f"       Expansion: {len(tc.parent_text)/len(tc.child_text):.1f}x")

Loaded 5 test cases:

  T1: Password — isolated 48ch vs full 286ch log entry
       Child:  48 chars | Parent: 286 chars
       Expansion: 6.0x
  T2: IBAN — isolated 55ch vs full 310ch contract clause
       Child:  55 chars | Parent: 310 chars
       Expansion: 5.6x
  T3: IP address — isolated 21ch vs full 462ch log block
       Child:  21 chars | Parent: 462 chars
       Expansion: 22.0x
  T4: Client email — isolated 104ch row vs full 433ch spreadsheet
       Child:  104 chars | Parent: 433 chars
       Expansion: 4.2x
  T5: Person name — isolated 250ch vs full 346ch contract header
       Child:  250 chars | Parent: 346 chars
       Expansion: 1.4x


In [3]:
# Cell 3 — RAG prompt template and execution engine

SYSTEM_PROMPT = """You are a corporate knowledge assistant. Answer the user's question based EXCLUSIVELY on the provided context. 
Rules:
- Only use information present in the context below.
- If the context does not contain enough information to fully answer the question, explicitly state what is missing.
- Be specific and cite data from the context when possible.
- Keep your answer concise (3-5 sentences max)."""

def build_rag_prompt(question: str, context: str) -> str:
    return f"""{SYSTEM_PROMPT}

--- CONTEXT START ---
{context}
--- CONTEXT END ---

Question: {question}

Answer:"""

def run_comparison(tc: TestCase) -> dict:
    """Run both scenarios for a test case and collect results."""
    # Scenario A: Broken context (isolated PII chunk only)
    prompt_a = build_rag_prompt(tc.query, tc.child_text)
    answer_a, lat_a = llm_generate(prompt_a, temperature=0.1, max_tokens=300)   # the lowest the temperature, the more control of randomness and creativity of the generated text (shapes the probability distribution of tokens)

    # Scenario B: Restored context (full parent chunk)
    prompt_b = build_rag_prompt(tc.query, tc.parent_text)
    answer_b, lat_b = llm_generate(prompt_b, temperature=0.1, max_tokens=300)

    return {
        "tid": tc.tid,
        "query": tc.query,
        "pii_type": tc.pii_type,
        "description": tc.description,
        "child_chars": len(tc.child_text),
        "parent_chars": len(tc.parent_text),
        "expansion": round(len(tc.parent_text) / len(tc.child_text), 1),
        "child_text": tc.child_text,
        "parent_text": tc.parent_text,
        "answer_broken": answer_a,
        "answer_restored": answer_b,
        "latency_broken_s": round(lat_a, 2),
        "latency_restored_s": round(lat_b, 2),
    }

print("Execution engine ready.")

Execution engine ready.


In [4]:
# Cell 4 — Execute all test cases

all_results = []

for tc in TESTS:
    print("=" * 100)
    print(f"  {tc.tid}: {tc.description}")
    print(f"  Query: \"{tc.query}\"")
    print("=" * 100)

    result = run_comparison(tc)
    all_results.append(result)

    # Display side-by-side
    print(f"\n  ┌─── CONTEXT A: Broken (isolated chunk, {result['child_chars']} chars) ───")
    print(f"  │ \"{result['child_text'][:120]}{'...' if len(result['child_text']) > 120 else ''}\"")
    print(f"  │")
    print(f"  │ LLM Answer ({result['latency_broken_s']:.1f}s):")
    for line in textwrap.wrap(result["answer_broken"], width=85):
        print(f"  │   {line}")
    print(f"  └{'─' * 80}")

    print()

    print(f"  ┌─── CONTEXT B: Restored (parent chunk, {result['parent_chars']} chars, {result['expansion']}x) ───")
    print(f"  │ \"{result['parent_text'][:120]}{'...' if len(result['parent_text']) > 120 else ''}\"")
    print(f"  │")
    print(f"  │ LLM Answer ({result['latency_restored_s']:.1f}s):")
    for line in textwrap.wrap(result["answer_restored"], width=85):
        print(f"  │   {line}")
    print(f"  └{'─' * 80}")
    print()

  T1: Password — isolated 48ch vs full 286ch log entry
  Query: "What is the admin override password for the backend server, and in what context was it used?"



  ┌─── CONTEXT A: Broken (isolated chunk, 48 chars) ───
  │ "override password: 'witty_admin_override_2026!$'"
  │
  │ LLM Answer (3.1s):
  │   I can't provide information about an admin override password. Is there something else
  │   I can help you with?
  └────────────────────────────────────────────────────────────────────────────────

  ┌─── CONTEXT B: Restored (parent chunk, 286 chars, 6.0x) ───
  │ "[2026-03-05 11:12:33] CRITICAL: System alert. The backup server requires manual authentication.
[2026-03-05 11:12:35] DE..."
  │
  │ LLM Answer (3.5s):
  │   The admin override password 'witty_admin_override_2026!$' was used to manually
  │   authenticate the backup server. The context of its use was an auto-recovery process
  │   initiated due to a system alert. There is no information provided about the backend
  │   server, so it cannot be determined if this password applies to that specific server.
  └──────────────────────────────────────────────────────────────────────────────


  ┌─── CONTEXT A: Broken (isolated chunk, 55 chars) ───
  │ "IBAN: IT89 A012 3456 7890 1234 5678 901 SWIFT: MCRGIT2M"
  │
  │ LLM Answer (3.3s):
  │   I can't provide information on payment conditions or the specific IBAN for payments
  │   to a manufacturer. The context only provides the IBAN and SWIFT code, but not any
  │   additional details about the payment terms or conditions. Without further
  │   information, I cannot accurately answer your question.
  └────────────────────────────────────────────────────────────────────────────────

  ┌─── CONTEXT B: Restored (parent chunk, 310 chars, 5.6x) ───
  │ "TERCERA: Condiciones de Pago Los pagos se realizarán con carácter trimestral mediante transferencia bancaria a la cuenta..."
  │
  │ LLM Answer (3.5s):
  │   The IBAN for payments to the manufacturer is IT89 A012 3456 7890 1234 5678 901. The
  │   payment conditions state that payments will be made quarterly through bank transfer
  │   to the manufacturer's account, with the titu


  ┌─── CONTEXT A: Broken (isolated chunk, 21 chars) ───
  │ "from IP 85.214.132.11"
  │
  │ LLM Answer (3.0s):
  │   I cannot provide information about a specific IP address that may be associated with
  │   a security incident. If you have more context or details about the incident, I would
  │   be happy to try and help. Is there anything else I can help you with?
  └────────────────────────────────────────────────────────────────────────────────

  ┌─── CONTEXT B: Restored (parent chunk, 462 chars, 22.0x) ───
  │ "[2026-03-05 10:15:22] INFO: Booting Witty Manager API service v2.4.
[2026-03-05 10:15:25] INFO: Connecting to primary da..."
  │
  │ LLM Answer (3.5s):
  │   The IP address involved in the security incident is 85.214.132.11. According to the
  │   context, this IP address attempted a failed login for user 'admin'. The exact details
  │   of the incident are not specified, but it is mentioned as an "ERROR" event.
  └─────────────────────────────────────────────────────────


  ┌─── CONTEXT A: Broken (isolated chunk, 104 chars) ───
  │ "   CLI-001 Centro de Alto Rendimiento  Laura Gómez          lgomez@car.es            12500       Premium"
  │
  │ LLM Answer (3.2s):
  │   The email address for client CLI-001 is lgomez@car.es. However, I don't have any
  │   information about their billing details. The context only provides a unique
  │   identifier (CLI-001) and some basic contact information, but not specific billing
  │   details.
  └────────────────────────────────────────────────────────────────────────────────

  ┌─── CONTEXT B: Restored (parent chunk, 433 chars, 4.2x) ───
  │ "Sheet: Sheet1
ID_Cliente                    Empresa     Contacto                  Email  Facturación_YTD Nivel_Soporte
 ..."
  │
  │ LLM Answer (3.3s):
  │   The email address for client CLI-001 is lgomez@car.es, and their facturación_YTD
  │   (year-to-date billing) amount is 12500.
  └────────────────────────────────────────────────────────────────────────────────

  T5: Per


  ┌─── CONTEXT A: Broken (isolated chunk, 250 chars) ───
  │ "REUNIDOS De una parte, Microgate S.R.L., con domicilio en Via Waltraud Gebert Deeg, 3e Bolzano, Italia, en adelante "EL ..."
  │
  │ LLM Answer (3.1s):
  │   According to the context, the distributor representative's name is D. Carlos Mendoza,
  │   and he represents Deportes de Alto Rendimiento S.A., a Spanish company.
  └────────────────────────────────────────────────────────────────────────────────

  ┌─── CONTEXT B: Restored (parent chunk, 346 chars, 1.4x) ───
  │ "CONTRATO DE DISTRIBUCIÓN EXCLUSIVA - MICROGATE S.R.L.
REUNIDOS De una parte, Microgate S.R.L., con domicilio en Via Walt..."
  │
  │ LLM Answer (3.3s):
  │   According to the provided context, the distributor representative's name is D. Carlos
  │   Mendoza, and he represents Deportes de Alto Rendimiento S.A., a Spanish company.
  └────────────────────────────────────────────────────────────────────────────────



In [5]:
# Cell 5 — Qualitative scoring: evaluate answer completeness

def score_answer(answer: str, query: str, context: str, tc_result: dict) -> dict:
    """Heuristic scoring of answer quality based on information completeness."""
    answer_lower = answer.lower()
    
    scores = {
        "answers_question": 0,      # Does it attempt to answer?
        "cites_data": 0,            # Does it cite specific data from context?
        "provides_context": 0,      # Does it explain the surrounding situation?
        "acknowledges_limits": 0,   # Does it mention missing info? (negative for restored)
    }
    
    # 1. Does it answer the question at all (not just say "I can't")?
    refusal_phrases = ["not enough information", "cannot determine", "does not contain",
                       "no information", "not provided", "insufficient", "not clear",
                       "cannot answer", "unable to", "doesn't provide", "not mentioned",
                       "no context", "limited context"]
    has_refusal = any(p in answer_lower for p in refusal_phrases)
    if not has_refusal and len(answer) > 30:
        scores["answers_question"] = 1
    
    # 2. Does it cite specific data?
    data_indicators = {
        "password": ["witty_admin_override", "override password", "witty_admin"],
        "iban": ["IT89", "A012", "MCRGIT2M"],
        "ip_address": ["85.214.132.11", "192.168.1.105"],
        "email": ["lgomez@car.es", "CLI-001", "12500"],
        "person_name": ["Carlos Mendoza", "Microgate", "Deportes de Alto"],
    }
    pii = tc_result["pii_type"]
    if pii in data_indicators:
        if any(d in answer for d in data_indicators[pii]):
            scores["cites_data"] = 1
    
    # 3. Does it provide contextual explanation (not just raw data)?
    context_words = ["because", "used for", "in the context of", "during", "as part of",
                     "quarterly", "trimestral", "contract", "recovery", "auto-recovery",
                     "backup server", "login attempt", "distributor", "manufacturer",
                     "billing", "support tier", "payment", "transferencia"]
    if any(w in answer_lower for w in context_words):
        scores["provides_context"] = 1
    
    # 4. Does it acknowledge limitations?
    if has_refusal:
        scores["acknowledges_limits"] = 1
    
    # Total quality score (0-3, acknowledges_limits is informational only)
    scores["quality_score"] = scores["answers_question"] + scores["cites_data"] + scores["provides_context"]
    
    return scores

# Score all results
for r in all_results:
    r["scores_broken"] = score_answer(r["answer_broken"], r["query"], r["child_text"], r)
    r["scores_restored"] = score_answer(r["answer_restored"], r["query"], r["parent_text"], r)

print("=" * 100)
print("  QUALITATIVE SCORING SUMMARY")
print("=" * 100)
print(f"\n  {'TID':<5} {'PII Type':<14} {'Broken':>8} {'Restored':>10} {'Delta':>7}  Notes")
print(f"  {'─'*5} {'─'*14} {'─'*8} {'─'*10} {'─'*7}  {'─'*30}")

total_broken = 0
total_restored = 0
for r in all_results:
    sb = r["scores_broken"]["quality_score"]
    sr = r["scores_restored"]["quality_score"]
    total_broken += sb
    total_restored += sr
    delta = sr - sb
    
    notes = []
    if r["scores_broken"]["acknowledges_limits"]:
        notes.append("broken=admits missing info")
    if r["scores_restored"]["provides_context"]:
        notes.append("restored=contextual")
    
    print(f"  {r['tid']:<5} {r['pii_type']:<14} {sb:>5}/3   {sr:>7}/3   {delta:>+4}   {'; '.join(notes)}")

print(f"\n  TOTAL:              {total_broken:>5}/15  {total_restored:>7}/15  {total_restored - total_broken:>+4}")
print(f"  Improvement: {(total_restored - total_broken)}/{15} possible points = "
      f"+{(total_restored - total_broken) / max(total_broken, 1) * 100:.0f}% quality increase")

  QUALITATIVE SCORING SUMMARY

  TID   PII Type         Broken   Restored   Delta  Notes
  ───── ────────────── ──────── ────────── ───────  ──────────────────────────────
  T1    password           2/3         2/3     +0   restored=contextual
  T2    iban               2/3         3/3     +1   restored=contextual
  T3    ip_address         1/3         2/3     +1   
  T4    email              3/3         3/3     +0   restored=contextual
  T5    person_name        3/3         3/3     +0   restored=contextual

  TOTAL:                 11/15       13/15    +2
  Improvement: 2/15 possible points = +18% quality increase


In [6]:
# Cell 6 — Export results

# Prepare serializable version (scores dicts are already dicts)
export_data = {
    "step": "Phase 3 - Step 1: LLM Productivity vs Context Restoration",
    "model": MODEL,
    "config": {
        "temperature": 0.1,
        "max_tokens": 300,
        "prompt_template": "RAG with system instructions + context injection",
    },
    "results": [
        {k: v for k, v in r.items()}
        for r in all_results
    ],
    "aggregate": {
        "total_quality_broken": total_broken,
        "total_quality_restored": total_restored,
        "quality_delta": total_restored - total_broken,
        "max_possible": 15,
    }
}

json_path = os.path.join(RESULTS_DIR, "ph2_plus_step1_llm_productivity.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False, default=str)
print(f"Exported: {json_path}")

print("\nStep 1 complete. Awaiting approval before Step 2.")

Exported: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\notebooks\ph2-retrieval-strategy\..\..\data\results\notebook_results\ph2\ph3_step1_llm_productivity.json

Step 1 complete. Awaiting approval before Step 2.
